In [169]:
%pip install docling


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Preload docling models

pip install -U "huggingface_hub[cli]"
huggingface-cli login
huggingface-cli download ds4sd/docling-models

In [170]:
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    WordFormatOption,
)
from docling.pipeline.simple_pipeline import SimplePipeline
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline

import logging
import os
import json
from dotenv import load_dotenv
from pathlib import Path
import glob


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

load_dotenv()
file_type=os.getenv("FILE_TYPE",".PDF")
file_source=os.getenv("FILE_SOURCE_LOCATION","/home/noelo/dev/instruct-injest/sourcedocs")
md_destination=os.getenv("MARKDOWN_LOCATION","/home/noelo/dev/instruct-injest/resultdocs")

if file_type.lower() not in ".pdf .docx .odf .txt":
    raise Exception("Invalid or empty file type. Only PDF, DOCX or ODF files supported. Set in FILE_TYPE envar")

if not file_source:
    raise Exception("Invalid or empty file source location. Set in FILE_SOURCE_LOCATION")

if not md_destination:
    raise Exception("Invalid or empty file source location. Set in MARKDOWN_LOCATION")

file_list=[]

Figure out what source of files we're dealing with and then list and filter them. Returning a list of files that we need to process.

In [171]:
def filter_file_ext(filename) -> bool:
    _, file_extension = os.path.splitext(filename)

    if not file_extension:
        return False
    
    if file_extension.lower().strip() in file_type.lower():
        return True
    else:
        return False

In [172]:
for file in glob.iglob(file_source+"/*", recursive=False):
    file_path = Path.joinpath(Path(file_source), file)
    file_list.append(file_path)
  
filtered_files = filter(filter_file_ext,file_list)

In [173]:
doc_converter = (
    DocumentConverter(  
        allowed_formats=[
            InputFormat.PDF,
            InputFormat.DOCX,
        ],  
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_cls=StandardPdfPipeline, backend=PyPdfiumDocumentBackend
            ),
            InputFormat.DOCX: WordFormatOption(
                pipeline_cls=SimplePipeline  
            ),
        },
    )
)

In [174]:
process_files=list(filtered_files)

conv_results = doc_converter.convert_all(
        process_files,
        raises_on_error=False, 
    )
out_path = Path(md_destination)

for res in conv_results:
    with (out_path / f"{res.input.file.stem}.md").open("wb") as fp:
                fp.write(res.document.export_to_markdown().encode("UTF-8"))

INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash 3d2abd0e021741887551c73bd132b421
INFO:docling.utils.accelerator_utils:Accelerator device: 'cuda:0'
INFO:docling.utils.accelerator_utils:Accelerator device: 'cuda:0'
INFO:docling.utils.accelerator_utils:Accelerator device: 'cuda:0'
INFO:docling.pipeline.base_pipeline:Processing document 2304.14953v2.pdf
INFO:docling.document_converter:Finished converting document 2304.14953v2.pdf in 9.56 sec.
